# Практическое занятие: Очистка текста от шума (Noise Cleaning)
## Кейс: Препроцессинг «грязных» текстовых данных из веб-источников (e-commerce / соцсети)

На этапе очистки шума мы удаляем элементы, которые не несут самостоятельной семантической нагрузки для классических NLP-моделей (или мешают им), но при этом **сохраняем структуру слов и пунктуацию для последующих этапов** (сегментации предложений и токенизации).

### Мы построим каскадный пайплайн очистки, который последовательно решает задачи:
1. Удаление HTML-тегов и декодирование HTML-сущностей (`&quot;`, `&amp;` и т.д.).
2. Удаление кликабельных URL-адресов и ссылок.
3. Удаление служебных системных символов и невидимых пробелов (Unicode-шум).
4. Нормализация пробельных символов (схлопывание лишних пробелов, табов и переносов строк).


In [1]:
import re
import html

# Эмулируем реальный "шумный" отзыв, собранный парсером с веб-страницы магазина
noisy_text = (
    "<p>Ужасный сервис!!!&#128544; Заказал товар на сайте <a href='https://example-shop.ru'>https://example-shop.ru</a>, "
    "но доставка опоздала на 3 дня.&nbsp;&nbsp;\n\n"
    "Менеджер (кажется, его звали проф. Иванов) постоянно говорил: &quot;Ждите...&quot; \t "
    "Связаться с поддержкой можно через Telegram-бота @shop_support_bot или мыло support@example.com!!!<br></p>"
)

print("=== ИСХОДНЫЙ ШУМНЫЙ ТЕКСТ ===")
print(noisy_text)

=== ИСХОДНЫЙ ШУМНЫЙ ТЕКСТ ===
<p>Ужасный сервис!!!&#128544; Заказал товар на сайте <a href='https://example-shop.ru'>https://example-shop.ru</a>, но доставка опоздала на 3 дня.&nbsp;&nbsp;

Менеджер (кажется, его звали проф. Иванов) постоянно говорил: &quot;Ждите...&quot; 	 Связаться с поддержкой можно через Telegram-бота @shop_support_bot или мыло support@example.com!!!<br></p>


## Декодирование HTML-сущностей и удаление HTML-тегов
Парсеры часто возвращают текст с HTML-тегами (`<p>`, `<br>`) и закодированными символами (например, `&#128544;` — это эмодзи 😠, а `&quot;` — кавычка). 
Используем стандартную библиотеку `html` для декодирования, а затем регулярное выражение для удаления тегов.


In [2]:
# 1. Декодируем сущности вроде &quot; -> " и &#128544; -> 😠
decoded_text = html.unescape(noisy_text)

# 2. Удаляем HTML-теги с помощью регулярного выражения
# Паттерн ищет текст внутри угловых скобок < ... >
html_pattern = re.compile(r'<[^>]+>')
clean_html = html_pattern.sub('', decoded_text)

print(clean_html)


Ужасный сервис!!!😠 Заказал товар на сайте https://example-shop.ru, но доставка опоздала на 3 дня.  

Менеджер (кажется, его звали проф. Иванов) постоянно говорил: "Ждите..." 	 Связаться с поддержкой можно через Telegram-бота @shop_support_bot или мыло support@example.com!!!


## Удаление URL-адресов и email
Ссылки и адреса почты содержат много уникальных токенов (`https`, `www`, `ru`), которые раздувают словарь модели и создают ложные корреляции. Заменяем их на пустую строку или специальный токен-заглушку (например, `[URL]`), если сам факт наличия ссылки важен для модели

In [3]:
# Паттерн для URL (поддерживает http, https, ftp, www)
url_pattern = re.compile(r'https?://\S+|www\.\S+')
# Паттерн для email
email_pattern = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b')

# Удаляем ссылки и почту
no_urls = url_pattern.sub('', clean_html)
no_emails = email_pattern.sub('', no_urls)

print(no_emails)


Ужасный сервис!!!😠 Заказал товар на сайте  но доставка опоздала на 3 дня.  

Менеджер (кажется, его звали проф. Иванов) постоянно говорил: "Ждите..." 	 Связаться с поддержкой можно через Telegram-бота @shop_support_bot или мыло !!!


## Удаление специфического веб-шума (юзернеймы, хештеги)
В зависимости от задачи, юзернеймы соцсетей (`@shop_support_bot`) также могут удаляться, чтобы модель не переобучалась на конкретные имена пользователей

In [4]:
# Паттерн для упоминаний в соцсетях/мессенджерах
mention_pattern = re.compile(r'@\w+')
no_mentions = mention_pattern.sub('', no_emails)

print(no_mentions)

Ужасный сервис!!!😠 Заказал товар на сайте  но доставка опоздала на 3 дня.  

Менеджер (кажется, его звали проф. Иванов) постоянно говорил: "Ждите..." 	 Связаться с поддержкой можно через Telegram-бота  или мыло !!!


## Нормализация пробелов и невидимых символов
После удаления тегов и ссылок в тексте остаются "дыры" из множественных пробелов, знаков табуляции `\t` и переносов строк `\n`. Схлопываем их до одного пробела.

In [5]:
# Паттерн \s+ находит любую последовательность пробелов, табов и переносов строк
spaces_pattern = re.compile(r'\s+')
final_clean_text = spaces_pattern.sub(' ', no_mentions).strip()

In [6]:
print("=== ИСХОДНЫЙ ШУМНЫЙ ТЕКСТ ===")
print(noisy_text)

print("=== РЕЗУЛЬТАТИВНЫЙ ОЧИЩЕННЫЙ ТЕКСТ ===")
print(final_clean_text)

=== ИСХОДНЫЙ ШУМНЫЙ ТЕКСТ ===
<p>Ужасный сервис!!!&#128544; Заказал товар на сайте <a href='https://example-shop.ru'>https://example-shop.ru</a>, но доставка опоздала на 3 дня.&nbsp;&nbsp;

Менеджер (кажется, его звали проф. Иванов) постоянно говорил: &quot;Ждите...&quot; 	 Связаться с поддержкой можно через Telegram-бота @shop_support_bot или мыло support@example.com!!!<br></p>
=== РЕЗУЛЬТАТИВНЫЙ ОЧИЩЕННЫЙ ТЕКСТ ===
Ужасный сервис!!!😠 Заказал товар на сайте но доставка опоздала на 3 дня. Менеджер (кажется, его звали проф. Иванов) постоянно говорил: "Ждите..." Связаться с поддержкой можно через Telegram-бота или мыло !!!
